# Phase 0 - composition partitioning consistency

Three experiments testing whether composition-based partitioning gives a consistent anatomy description across MRI acquisitions.

**No training.** No labels, no gradients, no GPU. This is inference and measurement on full volumes in patient coordinates.

Order of work, with reporting gates after steps 2 and 4:
1. Geometry + verification tests - nothing proceeds until these pass
2. Experiment 0, motion check, on 20 studies - **decision gate**
3. Tissue / composition / partition
4. Visual inspection on 5 studies
5. Consistency measurement
6. Experiments 1 and 2

Attach the competition dataset to this notebook before running.

## Setup

The Phase 0 modules, written out verbatim from the repo so the notebook is self-contained and still readable. Regenerate with `python scripts/make_kaggle_notebook.py` after any change.

In [ ]:
import os, sys
from pathlib import Path

if Path("/kaggle/working").exists():
    os.chdir("/kaggle/working")

PKG = Path.cwd() / "phase0"
for sub in ("composition", "experiments"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
    (PKG / sub / "__init__.py").write_text("")
if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
print("package root:", PKG)

In [ ]:
%%writefile phase0/composition/geometry.py
"""Voxel index -> patient coordinates, in millimetres.

Everything downstream depends on this being right.

Geometry is kept per-slice rather than collapsed into one 3D affine. RSNA
lumbar axial series are routinely acquired as per-level angled stacks, so a
single affine for the whole series is not merely imprecise, it is wrong.

Reference: DICOM PS3.3 C.7.6.2.1.1 (Image Plane Module).

DICOM index conventions, spelled out because they are easy to transpose:
  ImageOrientationPatient[0:3]  direction cosines of the first *row*. Walking
                               along a row means incrementing the COLUMN index.
  ImageOrientationPatient[3:6]  direction cosines of the first *column*.
                               Walking down a column increments the ROW index.
  PixelSpacing[0]              spacing between adjacent rows, i.e. the step
                               taken when the ROW index increments.
  PixelSpacing[1]              spacing between adjacent columns, i.e. the step
                               taken when the COLUMN index increments.

Patient coordinates are DICOM LPS: +x left, +y posterior, +z superior.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pydicom

LPS_AXES = ("x", "y", "z")
PLANE_BY_AXIS = {0: "sagittal", 1: "coronal", 2: "axial"}

# IOP rows are stored to 6-ish decimals; orthonormality holds to about 1e-5.
ORTHONORMAL_TOL = 1e-4


class GeometryError(ValueError):
    pass


@dataclass(frozen=True)
class SliceGeometry:
    """Plane geometry of a single DICOM instance."""

    instance_number: int
    position: np.ndarray  # (3,) ImagePositionPatient: centre of voxel (0, 0)
    row_cosine: np.ndarray  # (3,) IOP[0:3], traversed by incrementing the column index
    col_cosine: np.ndarray  # (3,) IOP[3:6], traversed by incrementing the row index
    row_spacing: float  # PixelSpacing[0], mm between adjacent rows
    col_spacing: float  # PixelSpacing[1], mm between adjacent columns
    thickness: float  # SliceThickness, mm
    rows: int
    cols: int
    path: str = ""

    def __post_init__(self):
        # Columns are the patient-space displacement per +1 of (row, col, normal).
        basis = np.stack([self.step_row, self.step_col, self.normal], axis=1)
        object.__setattr__(self, "_basis", basis)
        object.__setattr__(self, "_basis_inv", np.linalg.inv(basis))

    @property
    def step_row(self) -> np.ndarray:
        """Patient-space displacement per +1 row index."""
        return self.col_cosine * self.row_spacing

    @property
    def step_col(self) -> np.ndarray:
        """Patient-space displacement per +1 column index."""
        return self.row_cosine * self.col_spacing

    @property
    def normal(self) -> np.ndarray:
        return np.cross(self.row_cosine, self.col_cosine)

    def voxel_to_patient(self, row, col) -> np.ndarray:
        """Voxel centre(s) in patient mm. Scalars give (3,), arrays give (N, 3)."""
        row = np.asarray(row, dtype=float)
        col = np.asarray(col, dtype=float)
        return (
            self.position
            + row[..., None] * self.step_row
            + col[..., None] * self.step_col
        )

    def patient_to_voxel(self, point) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Inverse of voxel_to_patient.

        Returns (row, col, offset) as continuous values. `offset` is the signed
        distance from the point to this slice's plane, along the normal.

        Solved against the actual basis rather than by projection. Header
        cosines are orthogonal only to ~1e-4, and dot-product projection turns
        that into a round-trip error that grows with distance from the origin;
        inverting the basis is exact whatever the frame.
        """
        delta = np.asarray(point, dtype=float) - self.position
        coords = delta @ self._basis_inv.T
        row, col, offset = coords[..., 0], coords[..., 1], coords[..., 2]
        if row.ndim == 0:  # a single point in, plain floats out
            return float(row), float(col), float(offset)
        return row, col, offset

    def voxel_support_extent(self) -> np.ndarray:
        """Voxel size (mm) along (row axis, column axis, normal)."""
        return np.array([self.row_spacing, self.col_spacing, self.thickness])

    def support_axes(self) -> np.ndarray:
        """(3, 3) unit vectors matching voxel_support_extent, as patient directions."""
        return np.stack([self.col_cosine, self.row_cosine, self.normal])

    def corner_centres(self) -> np.ndarray:
        """(4, 3) centres of the four corner voxels."""
        rr = [0, 0, self.rows - 1, self.rows - 1]
        cc = [0, self.cols - 1, 0, self.cols - 1]
        return self.voxel_to_patient(np.array(rr), np.array(cc))

    def field_of_view(self) -> np.ndarray:
        """(2,) full in-plane extent (mm): matrix size x spacing, edge to edge."""
        return np.array([self.rows * self.row_spacing, self.cols * self.col_spacing])

    def contains(self, point, margin_mm: float = 0.0) -> bool:
        row, col, offset = self.patient_to_voxel(point)
        half = self.thickness / 2.0 + margin_mm
        return (
            -0.5 <= row <= self.rows - 0.5
            and -0.5 <= col <= self.cols - 0.5
            and abs(offset) <= half
        )


class SeriesGeometry:
    """Per-slice geometry for one DICOM series, ordered along the stack normal."""

    def __init__(self, slices: list[SliceGeometry], study_id=None, series_id=None):
        if not slices:
            raise GeometryError("series has no slices")
        self.study_id = study_id
        self.series_id = series_id
        self._slices = _sort_along_normal(slices)

    def __len__(self) -> int:
        return len(self._slices)

    def __getitem__(self, k: int) -> SliceGeometry:
        return self._slices[k]

    def __iter__(self):
        return iter(self._slices)

    @property
    def slices(self) -> list[SliceGeometry]:
        return self._slices

    @property
    def reference_normal(self) -> np.ndarray:
        """Mean slice normal, renormalised. Equals the slice normal when the
        stack is not angled."""
        stacked = np.stack([s.normal for s in self._slices])
        mean = stacked.mean(axis=0)
        norm = np.linalg.norm(mean)
        if norm < 1e-8:
            raise GeometryError("slice normals cancel; stack orientation is incoherent")
        return mean / norm

    @property
    def plane(self) -> str:
        return PLANE_BY_AXIS[int(np.argmax(np.abs(self.reference_normal)))]

    def voxel_to_patient(self, k: int, row, col) -> np.ndarray:
        return self._slices[k].voxel_to_patient(row, col)

    def voxel_centre(self, k: int, row, col) -> np.ndarray:
        return self._slices[k].voxel_to_patient(row, col)

    def voxel_support_extent(self, k: int) -> np.ndarray:
        return self._slices[k].voxel_support_extent()

    def slice_positions(self) -> np.ndarray:
        """(N,) projection of each slice origin onto the reference normal."""
        n = self.reference_normal
        return np.array([s.position @ n for s in self._slices])

    def slice_gaps(self) -> np.ndarray:
        return np.diff(self.slice_positions())

    def orientation_groups(self, tol: float = 1e-3) -> list[list[int]]:
        """Slice indices grouped by shared orientation. More than one group
        means the stack is angled, e.g. a per-level axial acquisition."""
        groups: list[list[int]] = []
        reps: list[np.ndarray] = []
        for k, s in enumerate(self._slices):
            iop = np.concatenate([s.row_cosine, s.col_cosine])
            for gi, rep in enumerate(reps):
                if np.allclose(iop, rep, atol=tol):
                    groups[gi].append(k)
                    break
            else:
                reps.append(iop)
                groups.append([k])
        return groups

    def is_angled(self, tol: float = 1e-3) -> bool:
        return len(self.orientation_groups(tol)) > 1

    def bounds(self) -> tuple[np.ndarray, np.ndarray]:
        """Axis-aligned patient-coordinate bounding box of the full voxel
        support, including the half-thickness beyond the first and last slice."""
        pts = []
        for s in self._slices:
            half_r = 0.5 * s.step_row
            half_c = 0.5 * s.step_col
            half_n = 0.5 * s.thickness * s.normal
            for corner in s.corner_centres():
                for dr in (-half_r, half_r):
                    for dc in (-half_c, half_c):
                        for dn in (-half_n, half_n):
                            pts.append(corner + dr + dc + dn)
        pts = np.asarray(pts)
        return pts.min(axis=0), pts.max(axis=0)

    def locate(self, point, margin_mm: float = 0.0):
        """Slice index whose plane the point falls in, or None if uncovered."""
        best = None
        best_offset = np.inf
        for k, s in enumerate(self._slices):
            row, col, offset = s.patient_to_voxel(point)
            if not (-0.5 <= row <= s.rows - 0.5 and -0.5 <= col <= s.cols - 0.5):
                continue
            if abs(offset) <= s.thickness / 2.0 + margin_mm and abs(offset) < best_offset:
                best, best_offset = (k, row, col, offset), abs(offset)
        return best

    def summary(self) -> dict:
        gaps = self.slice_gaps()
        s0 = self._slices[0]
        lo, hi = self.bounds()
        return {
            "study_id": self.study_id,
            "series_id": self.series_id,
            "plane": self.plane,
            "n_slices": len(self),
            "rows": s0.rows,
            "cols": s0.cols,
            "row_spacing_mm": s0.row_spacing,
            "col_spacing_mm": s0.col_spacing,
            "thickness_mm": s0.thickness,
            "median_gap_mm": float(np.median(gaps)) if gaps.size else float("nan"),
            "angled": self.is_angled(),
            "n_orientation_groups": len(self.orientation_groups()),
            "bounds_min": lo.tolist(),
            "bounds_max": hi.tolist(),
        }

    @classmethod
    def from_datasets(cls, datasets, study_id=None, series_id=None) -> "SeriesGeometry":
        return cls([slice_geometry(ds) for ds in datasets], study_id, series_id)

    @classmethod
    def from_dir(cls, directory, study_id=None, series_id=None) -> "SeriesGeometry":
        directory = Path(directory)
        files = sorted(directory.glob("*.dcm"), key=lambda p: int(p.stem))
        if not files:
            raise GeometryError(f"no DICOM files in {directory}")
        slices = []
        for f in files:
            ds = pydicom.dcmread(f, stop_before_pixels=True)
            slices.append(slice_geometry(ds, path=str(f)))
        if study_id is None:
            study_id = directory.parent.name
        if series_id is None:
            series_id = directory.name
        return cls(slices, study_id, series_id)


def slice_geometry(ds, path: str = "") -> SliceGeometry:
    """Build SliceGeometry from a pydicom dataset, validating the plane module."""
    for tag in ("ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing"):
        if not hasattr(ds, tag):
            raise GeometryError(f"{path or '<dataset>'}: missing {tag}")

    iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
    if iop.shape != (6,):
        raise GeometryError(f"{path}: ImageOrientationPatient must have 6 values")
    row_cosine, col_cosine = iop[:3], iop[3:]

    for name, vec in (("row", row_cosine), ("col", col_cosine)):
        if abs(np.linalg.norm(vec) - 1.0) > ORTHONORMAL_TOL:
            raise GeometryError(f"{path}: {name} direction cosine is not unit length")
    if abs(row_cosine @ col_cosine) > ORTHONORMAL_TOL:
        raise GeometryError(f"{path}: direction cosines are not orthogonal")

    # Headers store the cosines rounded to ~6 decimals, so they are a little
    # off unit length. Renormalise: the intent is a unit vector, and leaving
    # the rounding in place puts a ~1e-7 relative scale error on every
    # projection. Orthogonality is checked but not forced - squaring the frame
    # would overwrite what the scanner actually recorded.
    row_cosine = row_cosine / np.linalg.norm(row_cosine)
    col_cosine = col_cosine / np.linalg.norm(col_cosine)

    spacing = np.asarray(ds.PixelSpacing, dtype=float)
    thickness = float(getattr(ds, "SliceThickness", 0.0) or 0.0)
    if thickness <= 0:
        # Fall back to the reconstruction interval when thickness is absent.
        thickness = float(getattr(ds, "SpacingBetweenSlices", 0.0) or 0.0)
    if thickness <= 0:
        raise GeometryError(f"{path}: no usable SliceThickness or SpacingBetweenSlices")

    return SliceGeometry(
        instance_number=int(getattr(ds, "InstanceNumber", 0)),
        position=np.asarray(ds.ImagePositionPatient, dtype=float),
        row_cosine=row_cosine,
        col_cosine=col_cosine,
        row_spacing=float(spacing[0]),
        col_spacing=float(spacing[1]),
        thickness=thickness,
        rows=int(ds.Rows),
        cols=int(ds.Columns),
        path=path,
    )


def _sort_along_normal(slices: list[SliceGeometry]) -> list[SliceGeometry]:
    """Order by position along the stack normal, not by InstanceNumber.

    InstanceNumber ordering is a filesystem convention; position is physical.
    """
    stacked = np.stack([s.normal for s in slices])
    mean = stacked.mean(axis=0)
    norm = np.linalg.norm(mean)
    if norm < 1e-8:
        raise GeometryError("slice normals cancel; stack orientation is incoherent")
    n = mean / norm
    return sorted(slices, key=lambda s: float(s.position @ n))


def plane_trace(target: SliceGeometry, other: SliceGeometry):
    """Where `other`'s plane cuts across `target`'s image, in (row, col).

    Two non-parallel planes meet in a line. Drawing that line on the sagittal
    image shows exactly which vertebra an axial slice passes through - a direct
    check that needs no thresholds and no trust in a distance metric.

    Returns ((row0, col0), (row1, col1)), or None if the planes are parallel or
    the line misses the image.
    """
    n = other.normal
    # A point at (row, col) on `target` lies on `other`'s plane when
    #   c0 + c1*row + c2*col = 0
    c0 = (target.position - other.position) @ n
    c1 = target.step_row @ n
    c2 = target.step_col @ n

    lo_r, hi_r = -0.5, target.rows - 0.5
    lo_c, hi_c = -0.5, target.cols - 0.5
    hits = []
    if abs(c2) > 1e-12:
        for row in (lo_r, hi_r):
            col = -(c0 + c1 * row) / c2
            if lo_c <= col <= hi_c:
                hits.append((row, col))
    if abs(c1) > 1e-12:
        for col in (lo_c, hi_c):
            row = -(c0 + c2 * col) / c1
            if lo_r <= row <= hi_r:
                hits.append((row, col))

    if len(hits) < 2:
        return None
    # Keep the two furthest apart; edge cases can land the same corner twice.
    best, far = None, -1.0
    for i in range(len(hits)):
        for j in range(i + 1, len(hits)):
            d = np.hypot(hits[i][0] - hits[j][0], hits[i][1] - hits[j][1])
            if d > far:
                best, far = (hits[i], hits[j]), d
    return best if far > 1e-9 else None


def overlap_box(a: SeriesGeometry, b: SeriesGeometry):
    """Axis-aligned patient-space intersection of two series, or None."""
    lo_a, hi_a = a.bounds()
    lo_b, hi_b = b.bounds()
    lo = np.maximum(lo_a, lo_b)
    hi = np.minimum(hi_a, hi_b)
    return (lo, hi) if np.all(hi > lo) else None

In [ ]:
%%writefile phase0/composition/volume.py
"""Pixel data attached to per-slice geometry.

Slices are held as a list of 2D arrays, never stacked into a 3D block. Stacking
would presume a common lattice, and 18 of 25 axial series here are angled
per-level stacks for which no such lattice exists. Nothing in this module
resizes, resamples or interpolates: the anisotropy is the object of study.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pydicom

from .geometry import GeometryError, SeriesGeometry, slice_geometry


@dataclass
class RegionSample:
    """Voxels drawn from a patient-space neighbourhood."""

    intensities: np.ndarray  # (N,)
    coords: np.ndarray  # (N, 3) voxel centres, patient mm
    slice_index: np.ndarray  # (N,) which slice each voxel came from
    rows: np.ndarray  # (N,)
    cols: np.ndarray  # (N,)

    def __len__(self) -> int:
        return int(self.intensities.size)

    @property
    def n_slices(self) -> int:
        return int(np.unique(self.slice_index).size)

    def weighted_centroid(self, floor_at_zero: bool = True) -> np.ndarray:
        """Intensity-weighted centroid in patient mm.

        MR intensity has no calibrated zero, so a large constant pedestal drags
        this toward the plain geometric centre. That is a conservative failure
        for Exp 0 - it shrinks apparent discrepancy rather than inflating it -
        so the raw weighting the spec asks for is kept as the default.
        """
        w = self.intensities.astype(float)
        if floor_at_zero:
            w = np.clip(w, 0.0, None)
        total = w.sum()
        if total <= 0:
            raise ValueError("region has no positive intensity to weight by")
        return (w[:, None] * self.coords).sum(axis=0) / total

    def geometric_centroid(self) -> np.ndarray:
        return self.coords.mean(axis=0)


class Volume:
    """A series' pixel data, indexed by (slice, row, col) but addressed in mm."""

    def __init__(self, geometry: SeriesGeometry, pixels: list[np.ndarray]):
        if len(geometry) != len(pixels):
            raise GeometryError("geometry and pixel slice counts differ")
        for k, (g, p) in enumerate(zip(geometry, pixels)):
            if p.shape != (g.rows, g.cols):
                raise GeometryError(
                    f"slice {k}: pixels {p.shape} != header {(g.rows, g.cols)}"
                )
        self.geometry = geometry
        self.pixels = pixels

    def __len__(self) -> int:
        return len(self.geometry)

    @property
    def study_id(self):
        return self.geometry.study_id

    @property
    def series_id(self):
        return self.geometry.series_id

    @property
    def plane(self) -> str:
        return self.geometry.plane

    def intensity(self, k: int, row: int, col: int) -> float:
        return float(self.pixels[k][row, col])

    def slice_voxel_centres(self, k: int) -> np.ndarray:
        """(rows, cols, 3) patient coordinates of every voxel centre in slice k."""
        g = self.geometry[k]
        rr, cc = np.meshgrid(np.arange(g.rows), np.arange(g.cols), indexing="ij")
        return g.voxel_to_patient(rr, cc)

    def all_voxels(self) -> RegionSample:
        """Every voxel in the series, as a flat patient-coordinate sample."""
        return self._gather(range(len(self)), selector=None)

    def sample_sphere(self, centre_mm, radius_mm: float) -> RegionSample:
        """Voxels whose centres lie within radius_mm of a patient-space point.

        A slice contributes when the point is within radius_mm + half its
        thickness of the slice plane, so an anisotropic stack still supplies
        every slice that physically overlaps the sphere.
        """
        centre = np.asarray(centre_mm, dtype=float)
        return self._gather(range(len(self)), selector=("sphere", centre, radius_mm))

    def sample_box(self, centre_mm, half_extent_mm) -> RegionSample:
        centre = np.asarray(centre_mm, dtype=float)
        half = np.asarray(half_extent_mm, dtype=float)
        if half.ndim == 0:
            half = np.repeat(half, 3)
        return self._gather(range(len(self)), selector=("box", centre, half))

    def _gather(self, slice_indices, selector) -> RegionSample:
        vals, pts, ks, rs, cs = [], [], [], [], []

        for k in slice_indices:
            g = self.geometry[k]
            row_idx, col_idx = self._candidate_indices(g, selector)
            if row_idx is None:
                continue

            coords = g.voxel_to_patient(row_idx, col_idx)
            keep = self._refine(coords, selector)
            if keep is not None:
                if not keep.any():
                    continue
                row_idx, col_idx, coords = row_idx[keep], col_idx[keep], coords[keep]

            vals.append(self.pixels[k][row_idx, col_idx])
            pts.append(coords)
            ks.append(np.full(row_idx.shape, k))
            rs.append(row_idx)
            cs.append(col_idx)

        if not vals:
            empty_f = np.zeros(0)
            return RegionSample(empty_f, np.zeros((0, 3)), np.zeros(0, int),
                                np.zeros(0, int), np.zeros(0, int))
        return RegionSample(
            np.concatenate(vals).astype(float),
            np.concatenate(pts),
            np.concatenate(ks),
            np.concatenate(rs),
            np.concatenate(cs),
        )

    @staticmethod
    def _candidate_indices(g, selector):
        """Index-space bounding box for the selector, or (None, None) to skip."""
        if selector is None:
            rr, cc = np.meshgrid(np.arange(g.rows), np.arange(g.cols), indexing="ij")
            return rr.ravel(), cc.ravel()

        kind, centre, size = selector
        row_c, col_c, offset = g.patient_to_voxel(centre)
        reach = float(size) if kind == "sphere" else float(np.max(size))

        if abs(offset) > reach + g.thickness / 2.0:
            return None, None

        if kind == "sphere":
            # In-plane radius of the sphere's intersection with this plane.
            residual = max(0.0, size**2 - offset**2)
            in_plane = np.sqrt(residual)
            half_r = in_plane / g.row_spacing
            half_c = in_plane / g.col_spacing
        else:
            half_r = reach / g.row_spacing
            half_c = reach / g.col_spacing

        r0 = max(0, int(np.floor(row_c - half_r)))
        r1 = min(g.rows - 1, int(np.ceil(row_c + half_r)))
        c0 = max(0, int(np.floor(col_c - half_c)))
        c1 = min(g.cols - 1, int(np.ceil(col_c + half_c)))
        if r1 < r0 or c1 < c0:
            return None, None

        rr, cc = np.meshgrid(np.arange(r0, r1 + 1), np.arange(c0, c1 + 1), indexing="ij")
        return rr.ravel(), cc.ravel()

    @staticmethod
    def _refine(coords, selector):
        if selector is None:
            return None
        kind, centre, size = selector
        delta = coords - centre
        if kind == "sphere":
            return (delta**2).sum(axis=1) <= size**2
        return np.all(np.abs(delta) <= size, axis=1)

    @classmethod
    def from_dir(cls, directory, study_id=None, series_id=None) -> "Volume":
        directory = Path(directory)
        files = sorted(directory.glob("*.dcm"), key=lambda p: int(p.stem))
        if not files:
            raise GeometryError(f"no DICOM files in {directory}")

        entries = []
        for f in files:
            ds = pydicom.dcmread(f)
            entries.append((slice_geometry(ds, path=str(f)), _rescaled(ds)))

        geometry = SeriesGeometry(
            [g for g, _ in entries],
            study_id if study_id is not None else directory.parent.name,
            series_id if series_id is not None else directory.name,
        )
        # SeriesGeometry reorders along the normal; follow that ordering.
        by_path = {g.path: px for g, px in entries}
        return cls(geometry, [by_path[g.path] for g in geometry])


def _rescaled(ds) -> np.ndarray:
    """Pixel array with RescaleSlope/Intercept applied where present.

    Only ~1 in 4 of these series carries the rescale tags; the identity default
    is what the standard prescribes when they are absent.
    """
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    if slope != 1.0 or intercept != 0.0:
        arr = arr * slope + intercept
    return arr

In [ ]:
%%writefile phase0/experiments/exp0_motion.py
"""Experiment 0: does the patient move between the sagittal and axial acquisition?

For each study, a vertebral body centre is marked once in each acquisition. The
intensity-weighted centroid of a sphere around each mark is computed in that
acquisition's own voxels, converted to patient coordinates, and the two are
compared. If the discrepancy is a large fraction of the slice thickness, then
sagittal and axial are not describing the same anatomy in the same place, and
no amount of care in the composition model will make them agree.

Landmarks come from a CSV (see scripts/mark_landmarks.py):

    study_id,plane,series_id,instance_number,row,col
    109677683,sagittal,714837857,11,418,352
    109677683,axial,107963340,7,160,160

Usage:
    python experiments/exp0_motion.py --landmarks data/rsna/landmarks.csv
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from composition.volume import Volume  # noqa: E402

REQUIRED_COLUMNS = {"study_id", "plane", "series_id", "instance_number", "row", "col"}


def slice_index_for_instance(geometry, instance_number: int) -> int:
    for k, s in enumerate(geometry):
        if s.instance_number == int(instance_number):
            return k
    available = sorted(s.instance_number for s in geometry)
    raise KeyError(f"instance {instance_number} not in series (have {available})")


def landmark_centroid(volume: Volume, instance_number: int, row: int, col: int,
                      radius_mm: float, floor_at_zero: bool = True) -> dict:
    """Intensity-weighted centroid of a sphere seeded at a marked voxel."""
    k = slice_index_for_instance(volume.geometry, instance_number)
    seed = volume.geometry[k].voxel_to_patient(int(row), int(col))

    sample = volume.sample_sphere(seed, radius_mm)
    if len(sample) == 0:
        raise ValueError(f"no voxels within {radius_mm} mm of the mark")

    centroid = sample.weighted_centroid(floor_at_zero=floor_at_zero)
    return {
        "seed_mm": seed,
        "centroid_mm": centroid,
        "n_voxels": len(sample),
        "n_slices": sample.n_slices,
        "shift_from_seed_mm": float(np.linalg.norm(centroid - seed)),
        "thickness_mm": volume.geometry[k].thickness,
        "plane": volume.plane,
    }


def measure_study(sag_dir, ax_dir, sag_mark, ax_mark, radius_mm=8.0,
                  floor_at_zero=True) -> dict:
    sag = Volume.from_dir(sag_dir)
    axi = Volume.from_dir(ax_dir)

    s = landmark_centroid(sag, sag_mark["instance_number"], sag_mark["row"],
                          sag_mark["col"], radius_mm, floor_at_zero)
    a = landmark_centroid(axi, ax_mark["instance_number"], ax_mark["row"],
                          ax_mark["col"], radius_mm, floor_at_zero)

    delta = s["centroid_mm"] - a["centroid_mm"]
    discrepancy = float(np.linalg.norm(delta))
    thickest = max(s["thickness_mm"], a["thickness_mm"])

    return {
        "discrepancy_mm": discrepancy,
        "dx_mm": float(delta[0]),
        "dy_mm": float(delta[1]),
        "dz_mm": float(delta[2]),
        "sag_thickness_mm": s["thickness_mm"],
        "ax_thickness_mm": a["thickness_mm"],
        "ratio_to_thickest": discrepancy / thickest,
        "sag_centroid_mm": s["centroid_mm"].tolist(),
        "ax_centroid_mm": a["centroid_mm"].tolist(),
        "sag_n_voxels": s["n_voxels"],
        "ax_n_voxels": a["n_voxels"],
        "sag_n_slices": s["n_slices"],
        "ax_n_slices": a["n_slices"],
        "sag_shift_from_seed_mm": s["shift_from_seed_mm"],
        "ax_shift_from_seed_mm": a["shift_from_seed_mm"],
    }


def direction_consistency(vectors: np.ndarray) -> dict:
    """Are the discrepancy vectors pointing the same way?

    Uses the Rayleigh test for a preferred direction on the sphere: under
    isotropy 3*n*Rbar^2 is chi-squared with 3 degrees of freedom. An earlier
    version compared Rbar against a 3/sqrt(n) rule of thumb, which is far too
    conservative - it called Rbar=0.56 at n=25 "random" when the Rayleigh test
    puts it at p=3e-5.

    Alignment matters because patient motion is random per patient. A shared
    direction across subjects is a bias in the landmark or the geometry, and
    must be separated out before anything is called motion.
    """
    from scipy.stats import chi2

    norms = np.linalg.norm(vectors, axis=1)
    usable = norms > 1e-9
    if usable.sum() < 2:
        return {"n": int(usable.sum()), "resultant_length": float("nan")}

    units = vectors[usable] / norms[usable, None]
    mean_vec = units.mean(axis=0)
    resultant = float(np.linalg.norm(mean_vec))
    n = int(usable.sum())
    stat = 3.0 * n * resultant**2
    p = float(chi2.sf(stat, 3))
    return {
        "n": n,
        "resultant_length": resultant,
        "mean_direction": (mean_vec / resultant).tolist() if resultant > 1e-12 else None,
        "isotropic_expectation": float(1.0 / np.sqrt(n)),
        "rayleigh_stat": float(stat),
        "rayleigh_p": p,
        "looks_systematic": bool(p < 0.01),
    }


def decompose(vectors: np.ndarray) -> dict:
    """Split the discrepancy into a shared offset and per-study scatter.

    Exp 0 asks whether the patient moved. A component common to every study
    cannot be motion - it is a bias in how the landmark is identified in each
    plane, or in the geometry. The motion estimate is what remains after that
    offset is removed, so the decision rule belongs on the residual.

    The offset is the component-wise median, which shrugs off a bad mark.
    """
    systematic = np.median(vectors, axis=0)
    residual = vectors - systematic
    raw_mag = np.linalg.norm(vectors, axis=1)
    res_mag = np.linalg.norm(residual, axis=1)
    q1, q3 = np.percentile(res_mag, [25, 75])
    return {
        "systematic_mm": systematic.tolist(),
        "systematic_norm_mm": float(np.linalg.norm(systematic)),
        "residual_median_mm": float(np.median(res_mag)),
        "residual_iqr_mm": [float(q1), float(q3)],
        "residual_max_mm": float(res_mag.max()),
        "raw_median_mm": float(np.median(raw_mag)),
        "explained_fraction": float(
            1.0 - np.median(res_mag) / max(np.median(raw_mag), 1e-9)
        ),
        "residual_direction": direction_consistency(residual),
    }


def summarise(table: pd.DataFrame) -> dict:
    d = table["discrepancy_mm"].to_numpy()
    r = table["ratio_to_thickest"].to_numpy()
    vectors = table[["dx_mm", "dy_mm", "dz_mm"]].to_numpy()

    q1, q3 = np.percentile(d, [25, 75])
    rq1, rq3 = np.percentile(r, [25, 75])
    return {
        "n_studies": int(len(table)),
        "discrepancy_mm": {
            "median": float(np.median(d)), "iqr": [float(q1), float(q3)],
            "min": float(d.min()), "max": float(d.max()),
        },
        "ratio_to_thickest": {
            "median": float(np.median(r)), "iqr": [float(rq1), float(rq3)],
            "min": float(r.min()), "max": float(r.max()),
        },
        "per_axis_median_mm": {
            "x_LR": float(np.median(vectors[:, 0])),
            "y_AP": float(np.median(vectors[:, 1])),
            "z_SI": float(np.median(vectors[:, 2])),
        },
        "direction": direction_consistency(vectors),
        "decomposition": decompose(vectors),
        "residual_ratio_to_thickest": _residual_ratio(table, vectors),
    }


def _residual_ratio(table: pd.DataFrame, vectors: np.ndarray) -> dict:
    """Residual discrepancy as a fraction of the larger slice thickness."""
    thickest = np.maximum(table["sag_thickness_mm"].to_numpy(),
                          table["ax_thickness_mm"].to_numpy())
    res = np.linalg.norm(vectors - np.median(vectors, axis=0), axis=1) / thickest
    q1, q3 = np.percentile(res, [25, 75])
    return {"median": float(np.median(res)), "iqr": [float(q1), float(q3)],
            "max": float(res.max())}


def print_summary(summary: dict) -> None:
    d, r, ax, dirn = (summary["discrepancy_mm"], summary["ratio_to_thickest"],
                      summary["per_axis_median_mm"], summary["direction"])
    print(f"\nExperiment 0 - landmark discrepancy across {summary['n_studies']} studies")
    print(f"  discrepancy      median {d['median']:.2f} mm   "
          f"IQR [{d['iqr'][0]:.2f}, {d['iqr'][1]:.2f}]   range [{d['min']:.2f}, {d['max']:.2f}]")
    print(f"  / thickest slice median {r['median']:.2f}      "
          f"IQR [{r['iqr'][0]:.2f}, {r['iqr'][1]:.2f}]   range [{r['min']:.2f}, {r['max']:.2f}]")
    print(f"  per-axis median  x(L+) {ax['x_LR']:+.2f}  y(P+) {ax['y_AP']:+.2f}  "
          f"z(S+) {ax['z_SI']:+.2f} mm")

    if np.isnan(dirn.get("resultant_length", float("nan"))):
        print("  direction        too few usable vectors to assess")
        return
    verdict = "SYSTEMATIC" if dirn["looks_systematic"] else "no consistent direction"
    print(f"  direction        resultant {dirn['resultant_length']:.2f} "
          f"(isotropic ~{dirn['isotropic_expectation']:.2f}), Rayleigh "
          f"p={dirn['rayleigh_p']:.1e} -> {verdict}")
    if dirn.get("mean_direction"):
        m = dirn["mean_direction"]
        print(f"                   mean unit vector [{m[0]:+.2f}, {m[1]:+.2f}, {m[2]:+.2f}]")

    dec = summary.get("decomposition")
    if not dec:
        return
    sysv = dec["systematic_mm"]
    print(f"\n  Shared offset (cannot be motion - it is the same in every study)")
    print(f"    vector         x {sysv[0]:+.2f}   y {sysv[1]:+.2f}   z {sysv[2]:+.2f} mm"
          f"   |{dec['systematic_norm_mm']:.2f} mm|")
    print(f"    accounts for   {dec['explained_fraction']:.0%} of the raw median")
    rr = summary["residual_ratio_to_thickest"]
    print(f"\n  Residual after removing it - THIS is the motion estimate")
    print(f"    discrepancy    median {dec['residual_median_mm']:.2f} mm   "
          f"IQR [{dec['residual_iqr_mm'][0]:.2f}, {dec['residual_iqr_mm'][1]:.2f}]   "
          f"max {dec['residual_max_mm']:.2f}")
    print(f"    / thickest     median {rr['median']:.2f}   "
          f"IQR [{rr['iqr'][0]:.2f}, {rr['iqr'][1]:.2f}]")
    rd = dec["residual_direction"]
    if not np.isnan(rd.get("resultant_length", float("nan"))):
        left = "still SYSTEMATIC" if rd["looks_systematic"] else "isotropic, consistent with motion"
        print(f"    direction      Rayleigh p={rd['rayleigh_p']:.2f} -> {left}")


def load_landmarks(path: Path) -> pd.DataFrame:
    marks = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(marks.columns)
    if missing:
        raise SystemExit(f"{path}: landmark CSV is missing columns {sorted(missing)}")
    marks["plane"] = marks["plane"].str.lower().str.strip()
    bad = set(marks["plane"]) - {"sagittal", "axial"}
    if bad:
        raise SystemExit(f"{path}: unexpected plane values {sorted(bad)}")
    return marks


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--landmarks", required=True, help="CSV of marked vertebral body centres")
    parser.add_argument("--data_dir", default="data/rsna")
    parser.add_argument("--out", default="results/exp0_motion.csv")
    parser.add_argument("--radius_mm", type=float, default=8.0,
                        help="sphere radius around each mark; must stay inside the body")
    parser.add_argument("--raw_weights", action="store_true",
                        help="do not clip negative intensities before weighting")
    args = parser.parse_args()

    images = Path(args.data_dir) / "train_images"
    marks = load_landmarks(Path(args.landmarks))

    rows, skipped = [], []
    for study_id, group in marks.groupby("study_id"):
        planes = {p: g.iloc[0] for p, g in group.groupby("plane")}
        if not {"sagittal", "axial"} <= planes.keys():
            skipped.append((study_id, "needs both a sagittal and an axial mark"))
            continue

        sag_mark, ax_mark = planes["sagittal"], planes["axial"]
        sag_dir = images / str(study_id) / str(int(sag_mark.series_id))
        ax_dir = images / str(study_id) / str(int(ax_mark.series_id))

        try:
            result = measure_study(
                sag_dir, ax_dir,
                {"instance_number": sag_mark.instance_number, "row": sag_mark.row,
                 "col": sag_mark.col},
                {"instance_number": ax_mark.instance_number, "row": ax_mark.row,
                 "col": ax_mark.col},
                radius_mm=args.radius_mm,
                floor_at_zero=not args.raw_weights,
            )
        except (KeyError, ValueError, OSError) as err:
            skipped.append((study_id, str(err)))
            continue

        result["study_id"] = int(study_id)
        rows.append(result)
        print(f"  {study_id}: {result['discrepancy_mm']:6.2f} mm  "
              f"({result['ratio_to_thickest']:.2f} x thickest slice)")

    if skipped:
        print(f"\nSkipped {len(skipped)}:")
        for study_id, reason in skipped:
            print(f"  {study_id}: {reason}")

    if not rows:
        raise SystemExit("No studies measured.")

    table = pd.DataFrame(rows)
    lead = ["study_id", "discrepancy_mm", "sag_thickness_mm", "ax_thickness_mm",
            "ratio_to_thickest", "dx_mm", "dy_mm", "dz_mm"]
    table = table[lead + [c for c in table.columns if c not in lead]]

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out, index=False)

    summary = summarise(table)
    summary["radius_mm"] = args.radius_mm
    print_summary(summary)

    summary_path = out.with_suffix(".summary.json")
    summary_path.write_text(json.dumps(summary, indent=2))
    print(f"\nWrote {out}\nWrote {summary_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile phase0/experiments/exp0_retest.py
"""Test-retest: how reliably can the landmark be found at all?

Experiment 0 reported a 6.49 mm median discrepancy between the sagittal and
axial marks, decomposing into a ~4 mm shared anterior offset and ~5.3 mm of
scatter. That decomposition was fitted to two summary statistics, not measured.
This measures it.

A second, blind marking pass gives the repeatability of the landmark itself.
Because the Exp 0 residual and the retest difference are each the difference of
two independent marks, they are directly comparable - if marking noise is the
whole story, the retest scatter reproduces the Exp 0 residual with no fitting.

Per-axis scatter is the point, not just the magnitude:

  scatter isotropic, offset still A-P  -> the offset is a real difference in how
                                          the landmark reads in profile versus
                                          cross-section
  scatter itself A-P heavy             -> A-P is simply the hard axis to judge,
                                          and the "offset" is marking bias too

Usage:
    python experiments/exp0_retest.py --first landmarks.csv \
        --second landmarks_retest.csv --data_dir data/rsna
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from composition.volume import Volume  # noqa: E402
from experiments.exp0_motion import direction_consistency, landmark_centroid  # noqa: E402

AXES = ("x_LR", "y_AP", "z_SI")


def mark_deltas(first: pd.DataFrame, second: pd.DataFrame, images_dir,
                radius_mm: float = 8.0) -> pd.DataFrame:
    """Centroid difference between two marking passes, per study and plane."""
    images_dir = Path(images_dir)
    key = ["study_id", "plane"]
    a = first.set_index(key)
    b = second.set_index(key)
    shared = a.index.intersection(b.index)

    rows = []
    for idx in shared:
        study_id, plane = idx
        ra, rb = a.loc[idx], b.loc[idx]
        if int(ra.series_id) != int(rb.series_id):
            continue  # a different series is not a re-mark of the same thing
        volume = Volume.from_dir(images_dir / str(int(study_id)) / str(int(ra.series_id)))
        try:
            ca = landmark_centroid(volume, ra.instance_number, ra.row, ra.col, radius_mm)
            cb = landmark_centroid(volume, rb.instance_number, rb.row, rb.col, radius_mm)
        except (KeyError, ValueError):
            continue
        d = cb["centroid_mm"] - ca["centroid_mm"]
        rows.append({"study_id": int(study_id), "plane": plane,
                     "dx_mm": d[0], "dy_mm": d[1], "dz_mm": d[2],
                     "dist_mm": float(np.linalg.norm(d))})
    return pd.DataFrame(rows)


MAD_TO_SIGMA = 1.4826  # MAD -> sigma for a Gaussian
OUTLIER_MAD = 4.0      # robust z beyond which a re-mark is a gross error


def robust_sigma(v: np.ndarray) -> np.ndarray:
    """Per-axis SD estimated from the median absolute deviation.

    A single re-mark landing one vertebra away (~35 mm) sets the plain SD at
    n=20 to about 7.8 mm on its own. The SD then describes that one mistake
    rather than the repeatability of the landmark, so it cannot be the basis
    for a precision claim.
    """
    mad = np.median(np.abs(v - np.median(v, axis=0)), axis=0)
    return MAD_TO_SIGMA * mad


def scatter_summary(deltas: pd.DataFrame) -> dict:
    """Per-plane, per-axis repeatability, robust and non-robust side by side.

    `sigma_mark` is the noise on a single mark: a difference of two independent
    marks has sqrt(2) times the SD of one, so divide through.
    """
    out = {}
    for plane, grp in deltas.groupby("plane"):
        v = grp[["dx_mm", "dy_mm", "dz_mm"]].to_numpy()
        if len(v) < 2:
            continue
        sd = v.std(axis=0, ddof=1)
        rsd = robust_sigma(v)

        # Flag re-marks that are gross errors on any axis, then re-measure
        # without them so the scatter describes the landmark, not the mistakes.
        centre, scale = np.median(v, axis=0), np.maximum(rsd, 1e-6)
        z = np.abs(v - centre) / scale
        bad = np.any(z > OUTLIER_MAD, axis=1)
        clean = v[~bad]
        sd_clean = clean.std(axis=0, ddof=1) if len(clean) > 2 else sd

        use = rsd if np.all(rsd > 1e-6) else sd
        others = [use[0], use[2]]
        out[plane] = {
            "n": int(len(v)),
            "n_outliers": int(bad.sum()),
            "outlier_studies": grp.study_id.to_numpy()[bad].tolist(),
            "median_dist_mm": float(np.median(grp.dist_mm)),
            "sd_per_axis_mm": dict(zip(AXES, sd.round(3).tolist())),
            "sd_robust_per_axis_mm": dict(zip(AXES, rsd.round(3).tolist())),
            "sd_excl_outliers_mm": dict(zip(AXES, sd_clean.round(3).tolist())),
            "sigma_mark_per_axis_mm": dict(zip(AXES, (use / np.sqrt(2.0)).round(3).tolist())),
            "sigma_mark_sd_based_mm": dict(zip(AXES, (sd / np.sqrt(2.0)).round(3).tolist())),
            "contamination_ratio": float(np.max(sd / np.maximum(rsd, 1e-6))),
            "mean_per_axis_mm": dict(zip(AXES, v.mean(axis=0).round(3).tolist())),
            "median_per_axis_mm": dict(zip(AXES, np.median(v, axis=0).round(3).tolist())),
            "ap_anisotropy": float(use[1] / max(np.mean(others), 1e-9)),
            "ap_anisotropy_sd_based": float(sd[1] / max(np.mean([sd[0], sd[2]]), 1e-9)),
            "direction": direction_consistency(v),
        }
    return out


def predicted_between_plane(summary: dict) -> dict:
    """What Exp 0's residual would be if marking noise were the only cause.

    The sagittal and axial marks are independent, so their per-axis variances
    add. No fitting: this is a prediction the Exp 0 residual either matches or
    does not.
    """
    if not {"sagittal", "axial"} <= summary.keys():
        return {}
    s = np.array([summary["sagittal"]["sigma_mark_per_axis_mm"][a] for a in AXES])
    x = np.array([summary["axial"]["sigma_mark_per_axis_mm"][a] for a in AXES])
    per_axis = np.sqrt(s**2 + x**2)
    # Median magnitude of a zero-mean Gaussian with these per-axis SDs.
    rng = np.random.default_rng(0)
    draws = rng.normal(size=(20000, 3)) * per_axis
    return {
        "sd_per_axis_mm": dict(zip(AXES, per_axis.round(3).tolist())),
        "median_magnitude_mm": float(np.median(np.linalg.norm(draws, axis=1))),
    }


def interpret(summary: dict, predicted: dict, exp0_residual_median_mm=None,
              exp0_offset_mm=None) -> dict:
    """Turn the numbers into the two readings the protocol cares about."""
    notes = []
    aniso = {p: v["ap_anisotropy"] for p, v in summary.items()}
    ap_heavy = [p for p, r in aniso.items() if r > 1.5]

    # Contamination first: every reading below is meaningless if a handful of
    # gross errors are setting the scale.
    for plane, v in summary.items():
        if v.get("n_outliers"):
            notes.append(
                f"{plane}: {v['n_outliers']} of {v['n']} re-marks are gross errors "
                f"(studies {v['outlier_studies']}). Plain SD is "
                f"{v['contamination_ratio']:.1f}x the robust estimate, so the SD "
                "describes those mistakes, not the landmark. Robust figures used "
                "throughout; inspect those studies before trusting any of this.")
        elif v.get("contamination_ratio", 1.0) > 1.5:
            notes.append(
                f"{plane}: SD is {v['contamination_ratio']:.1f}x the robust estimate "
                "with no single re-mark flagged - the tail is heavy. Treat the "
                "precision figure as approximate.")

    if ap_heavy:
        notes.append(
            f"A-P scatter dominates in {', '.join(ap_heavy)} (ratio "
            f"{', '.join(f'{aniso[p]:.2f}' for p in ap_heavy)}). A-P is the hard "
            "axis to judge, so the shared offset is plausibly marking bias too, "
            "not a landmark-definition difference.")
    elif aniso:
        notes.append(
            "Scatter is close to isotropic (A-P ratio "
            f"{', '.join(f'{p}={r:.2f}' for p, r in aniso.items())}). If the Exp 0 "
            "offset remains A-P, it is a real difference between the views rather "
            "than an artefact of how the mark is placed.")

    verdict = None
    if exp0_residual_median_mm is not None and predicted:
        pred = predicted["median_magnitude_mm"]
        ratio = exp0_residual_median_mm / max(pred, 1e-9)
        if ratio < 1.3:
            verdict = ("marking noise explains the Exp 0 residual; motion is below "
                       "what this landmark can resolve")
        elif ratio < 2.0:
            verdict = ("marking noise explains most of the Exp 0 residual; any motion "
                       "is comparable to or smaller than the measurement precision")
        else:
            verdict = ("marking noise does NOT explain the Exp 0 residual; the excess "
                       "is a real between-acquisition difference")
        notes.append(f"observed residual {exp0_residual_median_mm:.2f} mm vs predicted "
                     f"{pred:.2f} mm from marking noise alone (ratio {ratio:.2f})")

    for plane, v in summary.items():
        # Median, not mean: a single level error drags the mean and would be
        # reported as a drift of the whole pass.
        drift = np.array([v["median_per_axis_mm"][a] for a in AXES])
        if np.linalg.norm(drift) > 0.5 * v["median_dist_mm"]:
            notes.append(f"{plane}: mean difference {drift.round(2).tolist()} mm is "
                         "large relative to the scatter - the second pass drifted, "
                         "which blind re-marking should have prevented.")
    return {"verdict": verdict, "notes": notes, "ap_anisotropy": aniso}


def print_report(deltas, summary, predicted, reading) -> None:
    print(f"\nTest-retest across {len(deltas)} (study, plane) re-marks")
    for plane, v in summary.items():
        sd, rsd, sm = (v["sd_per_axis_mm"], v["sd_robust_per_axis_mm"],
                       v["sigma_mark_per_axis_mm"])
        flag = f"  <-- {v['n_outliers']} gross error(s)" if v["n_outliers"] else ""
        print(f"\n  {plane}  (n={v['n']}){flag}")
        print(f"    median re-mark distance   {v['median_dist_mm']:.2f} mm")
        print("    SD per axis  (plain)      " +
              "  ".join(f"{a} {sd[a]:5.2f}" for a in AXES))
        print("    SD per axis  (robust)     " +
              "  ".join(f"{a} {rsd[a]:5.2f}" for a in AXES))
        if v["n_outliers"]:
            ex = v["sd_excl_outliers_mm"]
            print("    SD excluding outliers     " +
                  "  ".join(f"{a} {ex[a]:5.2f}" for a in AXES))
            print(f"    outlier studies           {v['outlier_studies']}")
        print("    sigma per mark (robust)   " +
              "  ".join(f"{a} {sm[a]:5.2f}" for a in AXES))
        print(f"    A-P anisotropy            {v['ap_anisotropy']:.2f} robust"
              f"  /  {v['ap_anisotropy_sd_based']:.2f} plain"
              "   (>1.5 means A-P is the hard axis)")
        print("    median difference (drift) " +
              "  ".join(f"{a} {v['median_per_axis_mm'][a]:+5.2f}" for a in AXES))

    if predicted:
        p = predicted["sd_per_axis_mm"]
        print("\n  Predicted Exp 0 residual from marking noise alone")
        print("    SD per axis               " +
              "  ".join(f"{a} {p[a]:5.2f}" for a in AXES))
        print(f"    median magnitude          {predicted['median_magnitude_mm']:.2f} mm")

    if reading.get("verdict"):
        print(f"\n  => {reading['verdict']}")
    for n in reading["notes"]:
        print(f"     - {n}")


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--first", required=True)
    parser.add_argument("--second", required=True)
    parser.add_argument("--data_dir", default="data/rsna")
    parser.add_argument("--out", default="results/exp0_retest.json")
    parser.add_argument("--radius_mm", type=float, default=8.0)
    parser.add_argument("--exp0_residual_mm", type=float, default=None,
                        help="residual median from the Exp 0 summary, for comparison")
    args = parser.parse_args()

    images = Path(args.data_dir) / "train_images"
    deltas = mark_deltas(pd.read_csv(args.first), pd.read_csv(args.second),
                         images, args.radius_mm)
    if deltas.empty:
        raise SystemExit("no (study, plane) pairs present in both passes")

    summary = scatter_summary(deltas)
    predicted = predicted_between_plane(summary)
    reading = interpret(summary, predicted, args.exp0_residual_mm)
    print_report(deltas, summary, predicted, reading)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({"summary": summary, "predicted": predicted,
                               "reading": reading}, indent=2))
    deltas.to_csv(out.with_suffix(".csv"), index=False)
    print(f"\nWrote {out}\nWrote {out.with_suffix('.csv')}")


if __name__ == "__main__":
    main()

In [ ]:
import composition.geometry, composition.volume, experiments.exp0_motion  # noqa
print("imported OK from", PKG)

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd

def find_data_dir():
    """Locate the competition wherever it got mounted.

    Attaching the competition gives /kaggle/input/<competition-slug>, but a
    community mirror lands under its own name and some are nested a level
    deeper. Search by content instead of assuming a path.

    Depth is capped deliberately: train_images holds ~2 million .dcm files, so
    an rglob would crawl for minutes.
    """
    roots = [Path("/kaggle/input"), Path("data/rsna"), Path("data"), Path(".")]
    patterns = ["train_series_descriptions.csv",
                "*/train_series_descriptions.csv",
                "*/*/train_series_descriptions.csv"]
    found = []
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            found += [h.parent for h in sorted(root.glob(pat))]
    # Prefer a directory that also carries the images.
    with_images = [d for d in found if (d / "train_images").is_dir()]
    return (with_images or found or [None])[0]

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    attached = sorted(p.name for p in Path("/kaggle/input").glob("*")) \
        if Path("/kaggle/input").exists() else []
    raise SystemExit(
        "Could not find train_series_descriptions.csv.\n"
        f"  Attached inputs: {attached or 'none'}\n"
        "  Add Input -> Competitions -> 'RSNA 2024 Lumbar Spine Degenerative "
        "Classification'.\n"
        "  You must have accepted the competition rules for it to appear."
    )

IMAGES = DATA_DIR / "train_images"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
WORK.mkdir(parents=True, exist_ok=True)

print("data :", DATA_DIR)
print("work :", WORK)
if not IMAGES.is_dir():
    print("\nWARNING: no train_images/ beside the CSVs - the DICOMs are not attached.")
else:
    print("studies on disk:", sum(1 for p in IMAGES.iterdir() if p.is_dir()))

## 1. Paired studies

Studies carrying both a sagittal T2/STIR and an axial T2 series. Where a study has several candidates, the one with the most label coordinates is taken - that is the series the graders read.

In [ ]:
SAG_T2, AXIAL_T2 = "Sagittal T2/STIR", "Axial T2"

series = pd.read_csv(DATA_DIR / "train_series_descriptions.csv")
coords = pd.read_csv(DATA_DIR / "train_label_coordinates.csv")
ann_counts = coords.groupby("series_id").size().to_dict()

def pick(rows):
    if rows.empty:
        return None
    return int(max(rows.series_id, key=lambda s: (ann_counts.get(s, 0), -s)))

pairs = []
for study_id, rows in series.groupby("study_id"):
    sag, axi = pick(rows[rows.series_description == SAG_T2]), pick(rows[rows.series_description == AXIAL_T2])
    if sag and axi:
        pairs.append({"study_id": int(study_id), "sag_series_id": sag, "ax_series_id": axi})

pairs = pd.DataFrame(pairs).sort_values("study_id").reset_index(drop=True)
print(f"studies with both a sagittal T2 and an axial T2: {len(pairs)}")

N_STUDIES = 25
selected = pairs.sample(n=min(N_STUDIES, len(pairs)), random_state=42).sort_values("study_id")
selected = selected.reset_index(drop=True)
selected.to_csv(WORK / "paired_studies.csv", index=False)
selected.head()

### Slice-count audit

The reason for moving here. A local Kaggle-API download with `--context_slices 1` yields only slices adjacent to a label coordinate: **3 sagittal slices** per study (p25 = p50 = p75 = 3), and axial stacks with gaps in 583/593 series. A 3-slice sagittal slab is ~15 mm thick, so a vertebral body's left-right centroid inside it is truncation-biased and Exp 0 would confound motion with the download window.

This cell confirms the mounted copy is complete.

In [ ]:
def slice_stats(df, col, label):
    counts, gapped = [], 0
    for r in df.itertuples():
        d = IMAGES / str(r.study_id) / str(getattr(r, col))
        inst = sorted(int(f.stem) for f in d.glob("*.dcm")) if d.is_dir() else []
        if not inst:
            continue
        counts.append(len(inst))
        gapped += (inst[-1] - inst[0] + 1) != len(inst)
    c = np.array(counts)
    print(f"{label:9s} n={len(c):3d}  slices min={c.min():3d} p25={np.percentile(c,25):5.1f} "
          f"median={np.median(c):5.1f} p75={np.percentile(c,75):5.1f} max={c.max():3d}   "
          f"non-contiguous={gapped}")
    return c

sag_counts = slice_stats(selected, "sag_series_id", "sagittal")
ax_counts  = slice_stats(selected, "ax_series_id",  "axial")

if np.median(sag_counts) <= 5:
    print("\nWARNING: sagittal series still look truncated - Exp 0 would be unreliable.")
else:
    print(f"\nFull series confirmed. Sagittal median {np.median(sag_counts):.0f} slices "
          f"(was 3 on the truncated local copy).")

## 2. Geometry verification

The three checks the spec requires, run on real paired series. Nothing downstream is trustworthy until these pass.

Geometry is per-slice, never one affine for the series: most axial lumbar stacks here are angled per disc level, so a single affine is wrong rather than merely imprecise.

In [ ]:
from composition.geometry import SeriesGeometry, overlap_box

RTOL, ATOL_MM = 1e-12, 1e-9
failures, rows = [], []

for r in selected.itertuples():
    sd, ad = IMAGES / str(r.study_id) / str(r.sag_series_id), IMAGES / str(r.study_id) / str(r.ax_series_id)
    if not (any(sd.glob("*.dcm")) and any(ad.glob("*.dcm"))):
        continue
    sag, axi = SeriesGeometry.from_dir(sd), SeriesGeometry.from_dir(ad)

    # check 1 - extent matches FOV x matrix size, corners coplanar
    for g in list(sag) + list(axi):
        c = g.corner_centres()
        if abs((c[2]-c[0]) @ g.col_cosine - (g.rows-1)*g.row_spacing) > RTOL * 1e3:
            failures.append((r.study_id, "extent-row"))
        if abs((c[1]-c[0]) @ g.row_cosine - (g.cols-1)*g.col_spacing) > RTOL * 1e3:
            failures.append((r.study_id, "extent-col"))
        if np.abs((c - g.position) @ g.normal).max() > ATOL_MM:
            failures.append((r.study_id, "coplanar"))

    # check 2 - one point resolved through two slices agrees
    for geom in (sag, axi):
        if len(geom) < 2:
            continue
        a, b = geom[0], geom[len(geom)-1]
        rng = np.random.default_rng(0)
        for _ in range(32):
            p = a.voxel_to_patient(rng.uniform(0, a.rows-1), rng.uniform(0, a.cols-1)) \
                + rng.uniform(-8, 8) * a.normal
            ra, ca, oa = a.patient_to_voxel(p); rb, cb, ob = b.patient_to_voxel(p)
            Ra = a.voxel_to_patient(ra, ca) + oa * a.normal
            Rb = b.voxel_to_patient(rb, cb) + ob * b.normal
            if max(np.linalg.norm(Ra-p), np.linalg.norm(Rb-p), np.linalg.norm(Ra-Rb)) > ATOL_MM:
                failures.append((r.study_id, "two-slice"))
                break

    # check 3 - the two acquisitions overlap in patient space
    box = overlap_box(sag, axi)
    if box is None or np.any(box[1] - box[0] <= 10.0):
        failures.append((r.study_id, "overlap"))

    ss, aa = sag.summary(), axi.summary()
    ext = (box[1] - box[0]) if box else np.zeros(3)
    rows.append(dict(study=r.study_id, sag_n=ss["n_slices"], ax_n=aa["n_slices"],
                     sag_th=ss["thickness_mm"], ax_th=aa["thickness_mm"],
                     ax_angled=aa["angled"], ax_groups=aa["n_orientation_groups"],
                     ovl_x=round(ext[0],1), ovl_y=round(ext[1],1), ovl_z=round(ext[2],1)))

audit = pd.DataFrame(rows)
print(f"checked {len(audit)} studies -> {'ALL PASS' if not failures else f'{len(failures)} FAILURES'}")
if failures:
    print(sorted(set(failures)))
print(f"angled axial stacks: {int(audit.ax_angled.sum())}/{len(audit)}")
audit.head(10)

### Measurement floor

One synthetic object imaged with two dissimilar geometries, zero motion by construction. Whatever discrepancy Exp 0 reports here is pure discretisation - the floor a real result has to clear.

In [ ]:
from composition.geometry import SliceGeometry
from composition.volume import Volume
from composition.geometry import SeriesGeometry
from experiments.exp0_motion import landmark_centroid

BLOB_CENTRE, BLOB_SIGMA = np.array([3.7, -41.3, -228.9]), 6.0

def phantom(plane, n, rs, cs, th, gap, rows=96, cols=96):
    if plane == "sagittal":
        rc, cc, stack = np.array([0.,1.,0.]), np.array([0.,0.,-1.]), np.array([1.,0.,0.])
    else:
        rc, cc, stack = np.array([1.,0.,0.]), np.array([0.,1.,0.]), np.array([0.,0.,1.])
    slices, pixels = [], []
    for k in range(n):
        origin = (BLOB_CENTRE + (k - (n-1)/2) * gap * stack
                  - cc * rs * (rows-1)/2 - rc * cs * (cols-1)/2)
        g = SliceGeometry(instance_number=k+1, position=origin, row_cosine=rc, col_cosine=cc,
                          row_spacing=rs, col_spacing=cs, thickness=th, rows=rows, cols=cols)
        rr, ccc = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")
        d2 = ((g.voxel_to_patient(rr, ccc) - BLOB_CENTRE)**2).sum(axis=-1)
        slices.append(g); pixels.append(np.exp(-d2/(2*BLOB_SIGMA**2)).astype(np.float32))
    return Volume(SeriesGeometry(slices, 1, 2), pixels)

sag_p, ax_p = phantom("sagittal",17,.55,.55,4.0,4.4), phantom("axial",21,.31,.31,3.5,3.8)
for R in (8.0, 12.0, 16.0):
    out = {}
    for name, v in (("sag", sag_p), ("ax", ax_p)):
        g = v.geometry[len(v)//2]
        rr, cc, _ = g.patient_to_voxel(BLOB_CENTRE)
        out[name] = landmark_centroid(v, g.instance_number, int(round(rr)), int(round(cc)), R)["centroid_mm"]
    print(f"R={R:>5} mm   floor = {np.linalg.norm(out['sag']-out['ax']):.4f} mm")

## 3. Experiment 0 - landmark marking

Mark the **same vertebral body** in both acquisitions - the body, not the disc, and near its centre in all three axes. L4 is a good default: mid-lumbar and well inside axial coverage in every study.

**Use the fallback marker below, not this one.** Click marking needs ipympl, whose frontend extension (`jupyter-matplotlib`) does not load on Kaggle - you get *Failed to load model class MPLCanvasModel* and no images. `pip install ipympl` does not fix it; the JS extension is never registered with the frontend. This cell is kept only for running elsewhere, e.g. local Jupyter.

The RSNA label coordinates cannot substitute for a manual mark: they sit on canal and subarticular points at *disc* levels, not body centres. They are drawn as faint blue crosses for orientation only.

In [ ]:
%matplotlib widget
import importlib.util

# Without ipympl the canvas model never registers and the frontend reports
# "Error displaying widget: model not found" - unhelpful, so say it plainly.
if importlib.util.find_spec("ipympl") is None:
    raise SystemExit(
        "ipympl is not installed, so click events cannot work.\n"
        "  Either: !pip install ipympl   then Run > Restart & Clear, re-run from cell 2\n"
        "  Or:     skip this cell and use the typing marker below (needs nothing extra)."
    )

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from composition.volume import Volume

LANDMARKS = WORK / "landmarks.csv"
COLUMNS = ["study_id", "plane", "series_id", "instance_number", "row", "col"]
N_TO_MARK = 20

def _hints(series_id):
    out = {}
    for r in coords[coords.series_id == series_id].itertuples():
        out.setdefault(int(r.instance_number), []).append((float(r.x), float(r.y)))
    return out

class Marker:
    """Slider + click marking for one series."""

    def __init__(self, volume, title, hints):
        self.v, self.hints, self.mark = volume, hints, None
        self.fig, self.ax = plt.subplots(figsize=(7, 7))
        self.fig.canvas.header_visible = False
        self.im = self.ax.imshow(volume.pixels[len(volume)//2], cmap="gray")
        self.dot, = self.ax.plot([], [], "o", color="#ff3b30", ms=9, mec="white", mew=1.2)
        self.crosses, = self.ax.plot([], [], "+", color="#4da3ff", ms=10, alpha=.55, ls="none")
        self.ax.set_axis_off(); self.title = title
        self.slider = widgets.IntSlider(value=len(volume)//2, min=0, max=len(volume)-1,
                                        description="slice", continuous_update=False)
        self.slider.observe(lambda c: self.show(c["new"]), names="value")
        self.fig.canvas.mpl_connect("button_press_event", self.click)
        self.show(self.slider.value)

    def show(self, k):
        self.k = k
        px, g = self.v.pixels[k], self.v.geometry[k]
        self.im.set_data(px); self.im.set_clim(np.percentile(px,1), np.percentile(px,99.5))
        h = self.hints.get(g.instance_number, [])
        self.crosses.set_data(*zip(*h)) if h else self.crosses.set_data([], [])
        self.ax.set_title(f"{self.title}\nslice {k+1}/{len(self.v)} (instance {g.instance_number})"
                          f"  -  {'marked' if self.mark else 'click the vertebral body centre'}",
                          fontsize=9)
        self.fig.canvas.draw_idle()

    def click(self, e):
        if e.inaxes is not self.ax or e.xdata is None:
            return
        g = self.v.geometry[self.k]
        r, c = int(round(e.ydata)), int(round(e.xdata))
        if 0 <= r < g.rows and 0 <= c < g.cols:
            self.mark = (self.k, r, c)
            self.dot.set_data([c], [r]); self.show(self.k)

    def result(self):
        if self.mark is None:
            return None
        k, r, c = self.mark
        return {"series_id": int(self.v.series_id),
                "instance_number": int(self.v.geometry[k].instance_number), "row": r, "col": c}

done = pd.read_csv(LANDMARKS) if LANDMARKS.exists() else pd.DataFrame(columns=COLUMNS)
complete = {s for s, g in done.groupby("study_id") if set(g.plane) >= {"sagittal", "axial"}}
todo = [r for r in selected.itertuples() if r.study_id not in complete][:N_TO_MARK]
print(f"{len(complete)} already marked, {len(todo)} to go")

state = {"i": 0, "markers": {}}
box = widgets.VBox([])
status = widgets.HTML()

def load(i):
    r = todo[i]
    state["markers"] = {}
    panels = []
    for plane, sid in (("sagittal", r.sag_series_id), ("axial", r.ax_series_id)):
        v = Volume.from_dir(IMAGES / str(r.study_id) / str(sid))
        m = Marker(v, f"{r.study_id} - {plane}", _hints(int(sid)))
        state["markers"][plane] = m
        panels.append(widgets.VBox([m.slider, widgets.Output()]))
        with panels[-1].children[1]:
            display(m.fig.canvas)   # ipympl renders the canvas, not the figure
    status.value = f"<b>[{i+1}/{len(todo)}] study {r.study_id}</b>"
    box.children = [status, widgets.HBox(panels), save_btn]

def save(_):
    global done
    got = {p: m.result() for p, m in state["markers"].items()}
    if any(v is None for v in got.values()):
        status.value += " &nbsp; <span style='color:#c00'>mark BOTH planes first</span>"
        return
    r = todo[state["i"]]
    fresh = pd.DataFrame([{**got[p], "study_id": int(r.study_id), "plane": p} for p in got])[COLUMNS]
    done = pd.concat([done, fresh]).drop_duplicates(subset=["study_id","plane"], keep="last")
    done.to_csv(LANDMARKS, index=False)
    plt.close("all")
    state["i"] += 1
    if state["i"] >= len(todo):
        box.children = [widgets.HTML(f"<b>Done - {len(done)//2} studies in {LANDMARKS}</b>")]
    else:
        load(state["i"])

save_btn = widgets.Button(description="Save & next", button_style="success")
save_btn.on_click(save)

if todo:
    load(0)
    display(box)
else:
    print("nothing to mark")

### Marker (use this one)

Inline images, core ipywidgets only - nothing that needs ipympl.

Set **slice** first, then type **row** / **col**; the crosshair follows as you type. The live readout under the panels converts both marks to patient coordinates and reports the gap, so a mismatch is visible before you save rather than after twenty studies.

Reading the gap: one lumbar vertebra is about 35 mm tall, so a gap near that size almost always means the two panes are on **different vertebrae**. That is the error worth catching - it would swamp the motion signal Exp 0 is trying to measure.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from composition.volume import Volume
from composition.geometry import SeriesGeometry, plane_trace
from experiments.exp0_motion import landmark_centroid

LANDMARKS = WORK / "landmarks.csv"
COLUMNS = ["study_id", "plane", "series_id", "instance_number", "row", "col"]
N_TO_MARK = 20
RADIUS_MM = 8.0          # must match the value cell 21 measures with

def _hints(series_id):
    out = {}
    for r in coords[coords.series_id == int(series_id)].itertuples():
        out.setdefault(int(r.instance_number), []).append((float(r.x), float(r.y)))
    return out

LEVELS = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

def disc_anchors(study_id, series_id):
    """World-space centre of each labelled disc level, from THIS series alone.

    Each acquisition is labelled by its own annotations. Pooling sagittal and
    axial points would assume they agree in space, which is the very thing
    Exp 0 is testing.
    """
    g = SeriesGeometry.from_dir(IMAGES / str(int(study_id)) / str(int(series_id)))
    by_inst = {sl.instance_number: sl for sl in g}
    rows = coords[(coords.study_id == int(study_id)) & (coords.series_id == int(series_id))]
    out = {}
    for lv, grp in rows.groupby("level"):
        pts = [by_inst[int(r.instance_number)].voxel_to_patient(r.y, r.x)
               for r in grp.itertuples() if int(r.instance_number) in by_inst]
        if pts:
            out[lv] = np.mean(pts, axis=0)
    return out

def level_of(point, anchors):
    """Name the level a point sits at, using the disc ladder along S-I."""
    have = [lv for lv in LEVELS if lv in anchors]
    if len(have) < 2:
        return "?"
    z = {lv: anchors[lv][2] for lv in have}
    ordered = sorted(have, key=lambda lv: -z[lv])
    pz = point[2]
    if pz > z[ordered[0]]:
        return "above " + ordered[0]
    if pz < z[ordered[-1]]:
        return "below " + ordered[-1]
    for a, b in zip(ordered, ordered[1:]):
        if z[b] <= pz <= z[a]:
            frac = (z[a] - pz) / max(z[a] - z[b], 1e-6)
            if frac <= 0.2:
                return a + " disc"
            if frac >= 0.8:
                return b + " disc"
            return b.split("/")[0] + " body"
    return "?"

def panel(volume, title, hints, on_change, overlay=None):
    n, g0 = len(volume), volume.geometry[0]
    sl = widgets.IntSlider(value=n // 2, min=0, max=n - 1, description="slice",
                           continuous_update=False)
    rw = widgets.IntText(value=g0.rows // 2, description="row")
    cl = widgets.IntText(value=g0.cols // 2, description="col")
    out = widgets.Output()

    def draw(*_):
        with out:
            clear_output(wait=True)
            k = sl.value
            px, g = volume.pixels[k], volume.geometry[k]
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.imshow(px, cmap="gray",
                      vmin=np.percentile(px, 1), vmax=np.percentile(px, 99.5))
            h = hints.get(g.instance_number, [])
            if h:
                ax.plot(*zip(*h), "+", color="#4da3ff", ms=10, alpha=.55, ls="none")
            ax.axhline(rw.value, color="#ff3b30", lw=.7)
            ax.axvline(cl.value, color="#ff3b30", lw=.7)
            ax.plot([cl.value], [rw.value], "o", color="#ff3b30", ms=8, mec="white", mew=1.2)

            # Where the current axial slice cuts this image. Two planes meet in
            # a line, so this shows which vertebra the axial slice passes
            # through - a direct check that needs no distance threshold.
            ov = overlay() if overlay else None
            if ov is not None:
                tr = plane_trace(g, ov)
                if tr is not None:
                    (r0, c0), (r1, c1) = tr
                    ax.plot([c0, c1], [r0, r1], "-", color="#ffd60a", lw=1.6, alpha=.9)
                    ax.text(c1, r1, f" axial {ov.instance_number}", color="#ffd60a",
                            fontsize=7, va="center", ha="right")

            ax.set_xticks(np.arange(0, g.cols, 50)); ax.set_yticks(np.arange(0, g.rows, 50))
            ax.tick_params(labelsize=6); ax.grid(color="#4da3ff", alpha=.25, lw=.4)
            ax.set_title(f"{title}\nslice {k+1}/{n} (instance {g.instance_number})", fontsize=9)
            plt.show()
            plt.close(fig)          # inline keeps every figure alive otherwise
        on_change()

    for w in (sl, rw, cl):
        w.observe(draw, names="value")
    draw()

    def get():
        return {"series_id": int(volume.series_id),
                "instance_number": int(volume.geometry[sl.value].instance_number),
                "row": int(rw.value), "col": int(cl.value)}

    def point_mm():
        # The Exp 0 quantity, not the raw voxel centre. A marked voxel is pinned
        # to its own slice plane, so comparing raw centres reports the gap
        # between the two planes rather than any real mismatch. The weighted
        # centroid averages over a sphere spanning several slices and escapes
        # that, which is exactly why Exp 0 uses it.
        return landmark_centroid(volume, volume.geometry[sl.value].instance_number,
                                 int(rw.value), int(cl.value), RADIUS_MM)["centroid_mm"]

    def current_slice():
        return volume.geometry[sl.value]

    return widgets.VBox([sl, rw, cl, out]), get, point_mm, draw, current_slice

done = pd.read_csv(LANDMARKS) if LANDMARKS.exists() else pd.DataFrame(columns=COLUMNS)
complete = {s for s, g in done.groupby("study_id") if set(g.plane) >= {"sagittal", "axial"}}
todo = [r for r in selected.itertuples() if r.study_id not in complete][:N_TO_MARK]
print(f"{len(complete)} already marked, {len(todo)} to go")

state = {"i": 0, "get": {}, "mm": {}, "anchors": {}}
shell, status, readout = widgets.VBox([]), widgets.HTML(), widgets.HTML()
save_btn = widgets.Button(description="Save & next", button_style="success")

def refresh(*_):
    if set(state["mm"]) < {"sagittal", "axial"}:
        return
    try:
        s, a = state["mm"]["sagittal"](), state["mm"]["axial"]()
    except Exception as err:
        readout.value = f"<span style='color:#c00'>centroid failed: {err}</span>"
        return
    d = s - a
    gap = float(np.linalg.norm(d))
    lv_s = level_of(s, state["anchors"].get("sagittal", {}))
    lv_a = level_of(a, state["anchors"].get("axial", {}))

    if lv_s != "?" and lv_a != "?" and lv_s != lv_a:
        colour, verdict = "#c00", f"DIFFERENT levels: {lv_s} vs {lv_a}"
    elif gap < 15:
        colour, verdict = "#1a7f37", f"good - both on {lv_s}"
    elif gap < 30:
        colour, verdict = "#9a6700", "same level, but drifting - recentre"
    else:
        colour, verdict = "#c00", "likely DIFFERENT vertebrae"

    sense = "inferior to" if d[2] < 0 else "superior to"
    readout.value = (
        f"<div style='font-family:monospace;font-size:13px'>"
        f"sagittal  {lv_s:<12s} ({s[0]:+7.1f}, {s[1]:+7.1f}, {s[2]:+7.1f}) mm<br>"
        f"axial     {lv_a:<12s} ({a[0]:+7.1f}, {a[1]:+7.1f}, {a[2]:+7.1f}) mm<br>"
        f"delta (sagittal - axial)  L {d[0]:+6.1f}   P {d[1]:+6.1f}   S {d[2]:+6.1f}<br>"
        f"<span style='color:#666'>sagittal mark is {abs(d[2]):.1f} mm {sense} the axial mark</span><br>"
        f"<b style='color:{colour}'>gap {gap:6.1f} mm - {verdict}</b>"
        f"<br><span style='color:#666'>weighted centroids, r={RADIUS_MM:.0f} mm - same measure cell 21 reports</span></div>")

def load(i):
    r = todo[i]
    # Sagittal is built first so it can trace whichever axial slice is current.
    vols = {p: Volume.from_dir(IMAGES / str(int(r.study_id)) / str(int(sid)))
            for p, sid in (("sagittal", r.sag_series_id), ("axial", r.ax_series_id))}
    sids = {"sagittal": r.sag_series_id, "axial": r.ax_series_id}

    w_s, get_s, mm_s, draw_s, _ = panel(
        vols["sagittal"], f"{r.study_id} - sagittal", _hints(int(sids["sagittal"])),
        refresh, overlay=lambda: state.get("ax_slice"))

    def axial_changed():
        state["ax_slice"] = state["ax_current"]() if state.get("ax_current") else None
        draw_s()                     # retrace the line on the sagittal pane
        refresh()

    w_a, get_a, mm_a, _, cur_a = panel(
        vols["axial"], f"{r.study_id} - axial", _hints(int(sids["axial"])), axial_changed)
    state["ax_current"] = cur_a
    state["ax_slice"] = cur_a()
    draw_s()

    panels = [w_s, w_a]
    state["get"] = {"sagittal": get_s, "axial": get_a}
    state["mm"] = {"sagittal": mm_s, "axial": mm_a}
    state["anchors"] = {p: disc_anchors(r.study_id, sids[p]) for p in sids}
    status.value = f"<b>[{i+1}/{len(todo)}] study {r.study_id}</b>"
    shell.children = [status, widgets.HBox(panels), readout, save_btn]
    refresh()

def save(_):
    global done
    r = todo[state["i"]]
    fresh = pd.DataFrame([{**state["get"][p](), "study_id": int(r.study_id), "plane": p}
                          for p in state["get"]])[COLUMNS]
    done = pd.concat([done, fresh]).drop_duplicates(subset=["study_id", "plane"], keep="last")
    done.to_csv(LANDMARKS, index=False)
    state["i"] += 1
    if state["i"] >= len(todo):
        shell.children = [widgets.HTML(f"<b>Done - {len(done)//2} studies in {LANDMARKS}</b>")]
    else:
        load(state["i"])

save_btn.on_click(save)
if todo:
    load(0); display(shell)
else:
    print("nothing left to mark")

## 4. Experiment 0 - measurement

Intensity-weighted centroid of a sphere around each mark, computed in each acquisition's own voxels, converted to patient coordinates and compared.

**Decision gate.** `phase0_experiment_protocol.md` is not in the repo, so absent the real rule this reports against *median discrepancy > 0.5 x the larger slice thickness => motion dominates, stop*. Replace `THRESHOLD` with the protocol's rule when available.

In [ ]:
from experiments.exp0_motion import measure_study, summarise, print_summary
import json

RADIUS_MM, THRESHOLD = 8.0, 0.5

if not LANDMARKS.exists():
    raise SystemExit(
        f"No landmarks saved yet at {LANDMARKS}.\n"
        "  Run cell 17 - or cell 19 if the widget one does not render - and mark\n"
        "  at least one study. The file is written when you press 'Save & next',\n"
        "  so nothing exists until a study is completed in BOTH planes."
    )

marks = pd.read_csv(LANDMARKS)
ready = [s for s, g in marks.groupby("study_id") if set(g.plane) >= {"sagittal", "axial"}]
if not ready:
    raise SystemExit(
        f"{LANDMARKS} has rows but no study is marked in both planes.\n"
        "  Exp 0 compares sagittal against axial, so each study needs both."
    )
print(f"{len(ready)} studies marked in both planes\n")

results, skipped = [], []

for study_id, g in marks.groupby("study_id"):
    planes = {p: sub.iloc[0] for p, sub in g.groupby("plane")}
    if not {"sagittal", "axial"} <= planes.keys():
        skipped.append((study_id, "needs both planes")); continue
    s, a = planes["sagittal"], planes["axial"]
    try:
        out = measure_study(
            IMAGES / str(study_id) / str(int(s.series_id)),
            IMAGES / str(study_id) / str(int(a.series_id)),
            {"instance_number": s.instance_number, "row": s.row, "col": s.col},
            {"instance_number": a.instance_number, "row": a.row, "col": a.col},
            radius_mm=RADIUS_MM)
    except Exception as err:
        skipped.append((study_id, str(err))); continue
    out["study_id"] = int(study_id); results.append(out)
    print(f"  {study_id}: {out['discrepancy_mm']:6.2f} mm  ({out['ratio_to_thickest']:.2f} x thickest)")

if skipped:
    print("\nskipped:", skipped)

table = pd.DataFrame(results)
table.to_csv(WORK / "exp0_motion.csv", index=False)
summary = summarise(table); summary["radius_mm"] = RADIUS_MM
print_summary(summary)

median_ratio = summary["ratio_to_thickest"]["median"]
verdict = "MOTION DOMINATES - stop and report" if median_ratio > THRESHOLD else "proceed to Stage 3"
summary["assumed_threshold"] = THRESHOLD
summary["verdict"] = verdict
print(f"\n>>> median ratio {median_ratio:.2f} vs assumed threshold {THRESHOLD} -> {verdict}")

(WORK / "exp0_motion.summary.json").write_text(json.dumps(summary, indent=2))

## 5. Test-retest - is the landmark even repeatable?

Exp 0's residual can only be called motion if the landmark can be found reliably. Measured sensitivity says otherwise: moving the mark 4 mm moves the centroid 3.8 mm (ratio 0.96), because vertebral marrow is near-uniform and an 8 mm sphere has no intensity structure to lock onto. Hand error passes through roughly 1:1.

So mark everything a second time and measure the repeatability directly.

**Blind means blind.** Do this on a different day, and do not open the first pass beforehand. This cell writes to a separate file, never loads the first one, and shuffles the study order so the sequence gives nothing away. Re-marking from memory measures recall, not the landmark.

Run the marker cell above first - this reuses its `panel()`.

In [ ]:
RETEST = WORK / "landmarks_retest.csv"

done_rt = pd.read_csv(RETEST) if RETEST.exists() else pd.DataFrame(columns=COLUMNS)
complete_rt = {s for s, g in done_rt.groupby("study_id")
               if set(g.plane) >= {"sagittal", "axial"}}

# Shuffle: marking in the same order as pass one is its own memory cue.
order = selected.sample(frac=1.0, random_state=1234).reset_index(drop=True)
todo_rt = [r for r in order.itertuples() if r.study_id not in complete_rt][:N_TO_MARK]
print(f"{len(complete_rt)} re-marked, {len(todo_rt)} to go   (writing to {RETEST.name})")

state_rt = {"i": 0, "get": {}, "mm": {}, "anchors": {}}
shell_rt, status_rt, readout_rt = widgets.VBox([]), widgets.HTML(), widgets.HTML()
save_rt = widgets.Button(description="Save & next", button_style="warning")

def refresh_rt(*_):
    if set(state_rt["mm"]) < {"sagittal", "axial"}:
        return
    try:
        sp, ap = state_rt["mm"]["sagittal"](), state_rt["mm"]["axial"]()
    except Exception as err:
        readout_rt.value = f"<span style='color:#c00'>{err}</span>"; return
    d = sp - ap
    lv_s = level_of(sp, state_rt["anchors"].get("sagittal", {}))
    lv_a = level_of(ap, state_rt["anchors"].get("axial", {}))
    ok = (lv_s == lv_a) and lv_s != "?"
    readout_rt.value = (
        f"<div style='font-family:monospace;font-size:13px'>"
        f"sagittal {lv_s} &nbsp; axial {lv_a}<br>"
        f"<b style='color:{'#1a7f37' if ok else '#c00'}'>"
        f"gap {np.linalg.norm(d):.1f} mm</b></div>")

def load_rt(i):
    r = todo_rt[i]
    sids = {"sagittal": r.sag_series_id, "axial": r.ax_series_id}
    vols = {p: Volume.from_dir(IMAGES / str(int(r.study_id)) / str(int(sid)))
            for p, sid in sids.items()}
    w_s, get_s, mm_s, draw_s, _ = panel(
        vols["sagittal"], f"{r.study_id} - sagittal", _hints(int(sids["sagittal"])),
        refresh_rt, overlay=lambda: state_rt.get("ax_slice"))

    def ax_changed():
        state_rt["ax_slice"] = state_rt["ax_current"]() if state_rt.get("ax_current") else None
        draw_s(); refresh_rt()

    w_a, get_a, mm_a, _, cur_a = panel(
        vols["axial"], f"{r.study_id} - axial", _hints(int(sids["axial"])), ax_changed)
    state_rt["ax_current"] = cur_a; state_rt["ax_slice"] = cur_a(); draw_s()
    state_rt["get"] = {"sagittal": get_s, "axial": get_a}
    state_rt["mm"] = {"sagittal": mm_s, "axial": mm_a}
    state_rt["anchors"] = {p: disc_anchors(r.study_id, sid) for p, sid in sids.items()}
    status_rt.value = (f"<b>[RETEST {i+1}/{len(todo_rt)}] study {r.study_id}</b>")
    shell_rt.children = [status_rt, widgets.HBox([w_s, w_a]), readout_rt, save_rt]
    refresh_rt()

def save_rt_fn(_):
    global done_rt
    r = todo_rt[state_rt["i"]]
    fresh = pd.DataFrame([{**state_rt["get"][p](), "study_id": int(r.study_id), "plane": p}
                          for p in state_rt["get"]])[COLUMNS]
    done_rt = pd.concat([done_rt, fresh]).drop_duplicates(subset=["study_id","plane"], keep="last")
    done_rt.to_csv(RETEST, index=False)
    state_rt["i"] += 1
    if state_rt["i"] >= len(todo_rt):
        shell_rt.children = [widgets.HTML(f"<b>Re-marked {len(done_rt)//2} studies</b>")]
    else:
        load_rt(state_rt["i"])

save_rt.on_click(save_rt_fn)
if todo_rt:
    load_rt(0); display(shell_rt)
else:
    print("nothing left to re-mark")

### Test-retest analysis

Per-axis scatter is the headline, not the magnitude. The two readings:

- **scatter isotropic, Exp 0 offset still A-P** -> the offset is a real difference between how the landmark reads in profile and in cross-section
- **scatter itself A-P heavy** -> A-P is simply the hard axis to judge, and the offset is marking bias too

The predicted residual is not fitted: sagittal and axial marks are independent, so their per-axis variances add, and Exp 0's residual either matches that prediction or exceeds it.

In [ ]:
from experiments.exp0_retest import (
    mark_deltas, scatter_summary, predicted_between_plane, interpret, print_report)
import json

first, second = WORK / "landmarks.csv", WORK / "landmarks_retest.csv"
if not second.exists():
    raise SystemExit(f"No second pass yet at {second} - run the retest marker above.")

deltas = mark_deltas(pd.read_csv(first), pd.read_csv(second), IMAGES, RADIUS_MM)
if deltas.empty:
    raise SystemExit("no (study, plane) pairs appear in both passes")

summary = scatter_summary(deltas)
predicted = predicted_between_plane(summary)

exp0 = json.loads((WORK / "exp0_motion.summary.json").read_text())     if (WORK / "exp0_motion.summary.json").exists() else {}
residual = (exp0.get("decomposition") or {}).get("residual_median_mm")

reading = interpret(summary, predicted, residual)
print_report(deltas, summary, predicted, reading)

deltas.to_csv(WORK / "exp0_retest.csv", index=False)
(WORK / "exp0_retest.json").write_text(json.dumps(
    {"summary": summary, "predicted": predicted, "reading": reading}, indent=2))
print()
print(f"Wrote {WORK/'exp0_retest.csv'} and {WORK/'exp0_retest.json'}")

---

**Stop here and report Exp 0 before building Stage 3.** The spec makes this a gate: if motion dominates, the composition model cannot rescue it.